In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import re
from transformers import AutoTokenizer, PreTrainedModel, AutoModel, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import DataCollatorWithPadding
import os

# 1. Cấu hình cơ bản & Mapping
MODEL_NAME = "../model/marbert_base"

# Mapping cho Stance
STANCE2ID = {"Against": 0, "Favor": 1, "None": 2}
# Mapping cho Sentiment (Giả sử có 3 nhãn)
SENTIMENT2ID = {"Negative": 0, "Positive": 1, "Neutral": 2}
# Mapping cho Sarcasm
SARCASM2ID = {"No": 0, "Yes": 1}

# 2. Tiền xử lý văn bản
def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"(.)\1+", r"\1\1", text)
    text = text.replace("#", " ")
    return re.sub(r"\s+", " ", text).strip()

def load_and_prep_multitask_data(file_path):
    df = pd.read_csv(file_path, keep_default_na=False)
    
    # Ép kiểu dữ liệu
    for col in ["target", "stance", "sentiment", "sarcasm", "text"]:
        df[col] = df[col].astype(str).str.strip()
        
    df["clean_text"] = df["text"].apply(clean_arabic_tweet)
    df["input_text"] = df["target"] + " [SEP] " + df["clean_text"]
    
    # Mapping nhãn. Lưu ý xử lý lỗi nếu dữ liệu thực tế có nhãn lạ
    df["label_stance"] = df["stance"].map(STANCE2ID).fillna(2).astype(int)
    # Giả định dữ liệu sentiment có các nhãn này, cần điều chỉnh theo thực tế tập dữ liệu
    df["label_sentiment"] = df["sentiment"].map(SENTIMENT2ID).fillna(2).astype(int) 
    df["label_sarcasm"] = df["sarcasm"].map(SARCASM2ID).fillna(0).astype(int)
    
    return df

print("Đang load dữ liệu cho Multi-Task...")
train_df = load_and_prep_multitask_data("../data/train.csv")
dev_df = load_and_prep_multitask_data("../data/dev.csv")

# Tính Class Weights cho Stance (như cũ)
stance_labels = train_df["label_stance"].tolist()
stance_weights = compute_class_weight(class_weight="balanced", classes=np.unique(stance_labels), y=stance_labels)
stance_weights_tensor = torch.tensor(stance_weights, dtype=torch.float)

# 3. Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_multitask(examples):
    tokenized = tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=128)
    # Trả về các cột nhãn tương ứng
    tokenized["labels_stance"] = examples["label_stance"]
    tokenized["labels_sentiment"] = examples["label_sentiment"]
    tokenized["labels_sarcasm"] = examples["label_sarcasm"]
    return tokenized

cols_to_keep = ["input_text", "label_stance", "label_sentiment", "label_sarcasm"]
train_dataset = Dataset.from_pandas(train_df[cols_to_keep]).map(tokenize_multitask, batched=True)
dev_dataset = Dataset.from_pandas(dev_df[cols_to_keep]).map(tokenize_multitask, batched=True)

# Gỡ bỏ các cột text vì Trainer không hiểu
cols_to_remove = ["input_text", "label_stance", "label_sentiment", "label_sarcasm"]
train_dataset = train_dataset.remove_columns(cols_to_remove)
dev_dataset = dev_dataset.remove_columns(cols_to_remove)

# 4. Định nghĩa Kiến trúc Multi-Task Model
class MultiTaskMARBERT(nn.Module):
    def __init__(self, model_name):
        super(MultiTaskMARBERT, self).__init__()
        # Load phần thân của MARBERT
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        
        # Tạo 3 cái đầu (heads) riêng biệt
        self.stance_head = nn.Linear(hidden_size, 3)     # 3 nhãn Stance
        self.sentiment_head = nn.Linear(hidden_size, 3)  # 3 nhãn Sentiment
        self.sarcasm_head = nn.Linear(hidden_size, 2)    # 2 nhãn Sarcasm

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # Chạy qua BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # Lấy vector đại diện [CLS]
        pooled_output = outputs.pooler_output 
        
        # Tính toán logits cho từng tác vụ
        logits_stance = self.stance_head(pooled_output)
        logits_sentiment = self.sentiment_head(pooled_output)
        logits_sarcasm = self.sarcasm_head(pooled_output)
        
        return logits_stance, logits_sentiment, logits_sarcasm

# 5. Custom Trainer cho Multi-Task (Đã tối ưu Label Smoothing)
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Lấy nhãn
        labels_stance = inputs.pop("labels_stance")
        labels_sentiment = inputs.pop("labels_sentiment")
        labels_sarcasm = inputs.pop("labels_sarcasm")
        
        # Chạy qua mô hình
        logits_stance, logits_sentiment, logits_sarcasm = model(**inputs)
        
        # Định nghĩa các hàm loss (Tích hợp Label Smoothing 0.1 cho Stance)
        loss_fct_stance = nn.CrossEntropyLoss(
            weight=stance_weights_tensor.to(model.bert.device), 
            label_smoothing=0.1  # <-- Điểm ăn tiền ở đây
        )
        loss_fct_sentiment = nn.CrossEntropyLoss()
        loss_fct_sarcasm = nn.CrossEntropyLoss()
        
        # Tính Loss cho từng phần
        loss_stance = loss_fct_stance(logits_stance, labels_stance)
        loss_sentiment = loss_fct_sentiment(logits_sentiment, labels_sentiment)
        loss_sarcasm = loss_fct_sarcasm(logits_sarcasm, labels_sarcasm)
        
        # HÀM LOSS TỔNG HỢP: Ưu tiên Stance (0.7), Sentiment (0.2), Sarcasm (0.1)
        total_loss = loss_stance + 0.2 * loss_sentiment + 0.1 * loss_sarcasm
        
        # Đóng gói đầu ra
        outputs = {"logits_stance": logits_stance}
        
        return (total_loss, outputs) if return_outputs else total_loss

# 6. Hàm đánh giá (Chỉ quan tâm đến Stance Favg2)
def compute_multitask_metrics(eval_pred):
    # Lấy logits của Stance (được đóng gói trong tuple)
    logits_tuple, labels_tuple = eval_pred 
    logits = logits_tuple[0] if isinstance(logits_tuple, tuple) else logits_tuple
    # Lấy labels của Stance (cột đầu tiên)
    labels = labels_tuple[0] if isinstance(labels_tuple, tuple) else labels_tuple
    
    predictions = np.argmax(logits, axis=-1)
    
    f_against = f1_score(labels, predictions, labels=[0], average="macro")
    f_favor = f1_score(labels, predictions, labels=[1], average="macro")
    favg2 = (f_favor + f_against) / 2.0
    return {"Favg2": favg2}

# 7. Huấn luyện (Đã tối ưu Siêu tham số)
model = MultiTaskMARBERT(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="../model/multitask_global_optimized", # Đổi tên thư mục để không đè file cũ
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    
    # --- CÁC THAM SỐ TỐI ƯU MỚI ---
    num_train_epochs=6,          # Tăng thời gian học để hội tụ hoàn toàn
    warmup_ratio=0.1,            # 10% steps đầu dùng để khởi động LR
    lr_scheduler_type="cosine",  # Giảm LR theo đường cong Cosine mượt mà
    weight_decay=0.01,           # Phạt trọng số lớn để chống Overfitting
    # ------------------------------
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False, 
    label_names=["labels_stance", "labels_sentiment", "labels_sarcasm"], 
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,      
    processing_class=tokenizer,       
    compute_metrics=compute_multitask_metrics,
)

print("Bắt đầu huấn luyện Multi-Task Bản Tối Ưu (6 Epochs)...")
trainer.train()

# Lưu mô hình thủ công
torch.save(model.state_dict(), "../model/best_multitask_global_optimized.pt")
print("Đã lưu mô hình Multi-Task Global (V2) tốt nhất!")

d:\StanceEval-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang load dữ liệu cho Multi-Task...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4499.31it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Bắt đầu huấn luyện Multi-Task Bản Tối Ưu (6 Epochs)...


Epoch,Training Loss,Validation Loss,Favg2
1,No log,1.129748,0.749197
2,No log,1.061609,0.764743
3,1.134833,1.059946,0.803092
4,1.134833,1.099452,0.823755
5,0.752612,1.121459,0.828248
6,0.752612,1.121179,0.824985


Đã lưu mô hình Multi-Task Global (V2) tốt nhất!
